# Is there a relationship between accessibility signal and distance to the TSS?

## opening ATAC data and filtering with the p value (jupyter_lotta)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
from pathlib import Path
ATAC_path = Path('data') / 'ImmGenATAC18_AllOCRsInfo.csv'
ATAC_data = pd.read_csv(ATAC_path)
ATAC_pfiltered = ATAC_data[ATAC_data['_-log10_bestPvalue'] > -np.log10(0.05)]
ATAC_pfiltered

### creating a file for the filtered ATACseq data

In [ ]:
ATAC_pfiltered.to_csv('data/ATAC_pfiltered.csv', index=False, header=True)
ATAC_pfiltered_check = pd.read_csv('data/ATAC_pfiltered.csv')
ATAC_pfiltered_check

## plot accessibility signal over distance to TSS

### open "Gene annotations"

In [ ]:
cols = [
    "gene_name",
    "transcript_name",
    "chrom",
    "strand",
    "txStart",
    "txEnd",
    "cdsStart",
    "cdsEnd",
    "exonCount",
    "exonStarts",
    "exonEnds"
]

gene_annotations_path = Path('data') / 'Gene annotations.txt'
gene_annotations = pd.read_csv(gene_annotations_path, sep="\t", names=cols)
gene_annotations.to_csv('data/gene_annotations.csv', index=False)
gene_annotations_check = pd.read_csv('data/gene_annotations.csv')
gene_annotations_check

### Match ATAC genes to gene annotations and compute distance to TSS

In [ ]:
gene_and_TSS = gene_annotations[["gene_name", "txStart"]].copy()


ATAC_merged = ATAC_pfiltered.merge(
    gene_and_TSS,
    left_on="genes.within.100Kb",
    right_on="gene_name",
    how="right"
)


ATAC_merged["distance_to_tss"] = abs(ATAC_merged["Summit"] - ATAC_merged["txStart"])


keep_cols = list(ATAC_pfiltered.columns[8:]) + ["distance_to_tss"]
ATAC_dis_TSS = ATAC_merged[keep_cols].copy()

ATAC_dis_TSS

### creating a scatter plot connecting accessibility score to distance from TSS

In [ ]:
plot_data = ATAC_dis_TSS.iloc[:, :-1]
x = ATAC_dis_TSS['distance_to_tss']

plt.figure(figsize=(15, 10))
for col in plot_data.columns:
    plt.scatter(x, plot_data[col], alpha=0.25, s=8)
plt.xlabel('Distance to TSS')
plt.ylabel('ATAC value')
plt.title('ATAC values vs. distance to TSS (all ATAC columns)')
plt.axvline(0, color='red', linestyle='--', linewidth=1, label='TSS')
plt.axvline(ATAC_dis_TSS['distance_to_tss'].median(), color='red', linestyle='--', linewidth=1, label='Median distance to TSS')
plt.legend()
plt.show()


There seems to be a problem with this plot. Distance to TSS should not be greater than 100.000.
Since the median and the mean are very low compared to the highest values one possible solution coul be to just remove very high values.

In [ ]:
ATAC_dis_TSS['distance_to_tss'].quantile(0.99)

Remove the top 1 % of values

In [ ]:
ATAC_dis_TSS_filtered = ATAC_dis_TSS.drop(ATAC_dis_TSS[ATAC_dis_TSS['distance_to_tss'] > ATAC_dis_TSS['distance_to_tss'].quantile(0.99)].index)
ATAC_dis_TSS_filtered

In [ ]:
plot_data_filtered = ATAC_dis_TSS_filtered.iloc[:, :-1]
x = ATAC_dis_TSS_filtered['distance_to_tss']

plt.figure(figsize=(15, 10))
for col in plot_data_filtered.columns:
    plt.scatter(x, plot_data_filtered[col], alpha=0.25, s=8, color='steelblue')
plt.xlabel('Distance to TSS')
plt.ylabel('ATAC value')
plt.title('ATAC values vs. distance to TSS (all ATAC columns)')
plt.axvline(0, color='red', linestyle='--', linewidth=1, label='TSS')
plt.axvline(ATAC_dis_TSS_filtered['distance_to_tss'].median(), color='red', linestyle='--', linewidth=1, label='Median distance to TSS')
plt.legend()
plt.show()

for a max distance of 100.000

In [ ]:
ATAC_dis_TSS_100kb = ATAC_dis_TSS.drop(ATAC_dis_TSS[ATAC_dis_TSS['distance_to_tss'] > 100000].index)

plot_data_100kb = ATAC_dis_TSS_100kb.iloc[:, :-1]
x = ATAC_dis_TSS_100kb['distance_to_tss']

plt.figure(figsize=(15, 10))
for col in plot_data_100kb.columns:
    plt.scatter(x, plot_data_100kb[col], alpha=0.25, s=8, color='steelblue')
plt.xlabel('Distance to TSS')
plt.ylabel('ATAC value')
plt.title('ATAC values vs. distance to TSS (all ATAC columns)')
plt.axvline(0, color='red', linestyle='--', linewidth=1, label='TSS')
plt.axvline(ATAC_dis_TSS_100kb['distance_to_tss'].median(), color='red', linestyle='--', linewidth=1, label='Median distance to TSS')
plt.legend()
plt.show()

## creating a histogram of active OCRs in relation to distance from TSS

In [ ]:
ATAC_dis_TSS_filtered.describe()

In [ ]:
ATAC_value_median = ATAC_dis_TSS_filtered.iloc[:, :-1].median(axis=None)

In [ ]:
atac_values = ATAC_dis_TSS_filtered.iloc[:, :-1]

active_atac_counts = (atac_values.fillna(0) > ATAC_value_median).sum(axis=1)

distances = ATAC_dis_TSS_filtered['distance_to_tss']

plt.figure(figsize=(10, 7))
plt.hist(distances, bins=50, weights=active_atac_counts)
plt.xlabel('Distance to TSS')
plt.ylabel('Number of above-median ATAC values per row')
plt.title('histogram: ATAC count vs distance to TSS')
plt.show()

the drop of at 100kb could be due to the fact that there should not be any values above 100kb in the first place

creating the same histogram with only those distances of up to 100kb

In [ ]:
atac_values = ATAC_dis_TSS_100kb.iloc[:, :-1]

active_atac_counts = (atac_values.fillna(0) > ATAC_value_median).sum(axis=1)

distances = ATAC_dis_TSS_100kb['distance_to_tss']

plt.figure(figsize=(10, 7))
plt.hist(distances, bins=20, weights=active_atac_counts)
plt.xlabel('Distance to TSS')
plt.ylabel('Number of above-median ATAC values per row')
plt.title('histogram: ATAC count vs distance to TSS up to 100kb')
plt.savefig('tilmann_plots/histogram_ATAC_count_vs_distance_to_TSS_100kb.png', dpi=300, bbox_inches="tight")
plt.show()

### quantifying the relationship between accessibility score and distance to TSS

In [ ]:
counts, bin_edges, patches = plt.hist(distances, bins=20, weights=active_atac_counts)
print(counts[0])
print(counts[1:])
print(counts[1:].mean())
print(counts[1:].std())

In [ ]:
from scipy import stats

first_bin_count = counts[0]
other_bin_counts = counts[1:]

ttest_result = stats.ttest_1samp(other_bin_counts, popmean=first_bin_count)

print('First bin count:', first_bin_count)
print('Other bins counts:', other_bin_counts)
print('Other bins mean:', other_bin_counts.mean())
print('One-sample t-test result:', ttest_result)
print('p-value:', ttest_result.pvalue)


In [ ]:
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
other_bin_centers = bin_centers[1:]
other_bin_counts = counts[1:]

linreg = stats.linregress(other_bin_centers, other_bin_counts)

print('Linear regression (excluding first bin):')
print('Slope:', linreg.slope)
print('R-squared:', linreg.rvalue ** 2)
print('p-value:', linreg.pvalue)

plt.figure(figsize=(10, 7))
plt.scatter(other_bin_centers, other_bin_counts, label='Bin counts', color='steelblue')
plt.plot(other_bin_centers, linreg.intercept + linreg.slope * other_bin_centers, color='red', label='Linear fit')
plt.xlabel('Distance bin center (bp)')
plt.ylabel('Count of above-median ATAC values')
plt.title('Linear regression on counts for bins 2-20')
plt.savefig('tilmann_plots/linear_regression_ATAC_count_vs_distance_to_TSS_100kb_no_Promotors.png', dpi=300, bbox_inches="tight")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
predicted_counts = linreg.intercept + linreg.slope * other_bin_centers
residuals = other_bin_counts - predicted_counts
print('Residuals mean:', np.mean(residuals))
print('Residuals std:', np.std(residuals, ddof=1))
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(residuals, bins=15, color='steelblue', edgecolor='black')
plt.axvline(0, color='red', linestyle='--')
plt.xlabel('Residual')
plt.ylabel('Frequency')
plt.title('Histogram of residuals')
plt.subplot(1, 2, 2)
stats.probplot(residuals, dist='norm', plot=plt)
plt.title('QQ plot of residuals')
plt.tight_layout()
plt.show()


## Looking at the OCRs with very high distances to the TSS

In [ ]:

gene_and_TSS = gene_annotations[["gene_name", "txStart"]].copy()


ATAC_merged = ATAC_pfiltered.merge(
    gene_and_TSS,
    left_on="genes.within.100Kb",
    right_on="gene_name",
    how="right"
)


ATAC_merged["distance_to_tss"] = abs(ATAC_merged["Summit"] - ATAC_merged["txStart"])

In [ ]:
ATAC_dis_TSS_highest_with_na = ATAC_merged.drop(ATAC_merged[ATAC_merged['distance_to_tss'] < ATAC_merged['distance_to_tss'].quantile(0.999)].index)
ATAC_dis_TSS_highest = ATAC_dis_TSS_highest_with_na.dropna(subset=["distance_to_tss"]).copy()
ATAC_dis_TSS_highest

In [ ]:
plt.figure(figsize=(10, 7))
plt.hist(ATAC_dis_TSS_highest["distance_to_tss"], bins=200,)
plt.xlabel('Distance to TSS')
plt.ylabel('Number of ATAC Peaks with that distance')
plt.title('histogram: distribution of high distances to tss')
plt.show()

In [ ]:
ATAC_dis_TSS_highest_filtered = ATAC_dis_TSS_highest.drop(
    ATAC_dis_TSS_highest[
        ATAC_dis_TSS_highest[ATAC_dis_TSS_highest.columns[8:97]].max(axis=1) < 40
    ].index)
ATAC_dis_TSS_highest_filtered

most OCRs with high accessibility scores and large distances from the TSS seem to be conetcted to similar genes:

- "Mir" genes are coding for microRNA which has regulatory functions
- I could not find any "Bc1" genes on the X-chromosome
- some of the OCRs lie inside of other genes, therefore it is not likely that they are functionally connected to the gene from the data frame